In [1]:
## Fill this cell and run whole notebook

## Source and destionation parameters
read_path_pattern = 'Files/ingestion'
# export_table_raw = 'Tables/dbo/cost_raw'  # output path for your delta parquet table
export_table_final = 'Tables/dbo/cost_details'  # output path for your delta parquet table
export_table_calendar = 'Tables/dbo/calendar'  # output path for calendar table

## Tags that needs to extracted to seperate columns
PromotedTags = {"Application", "BusinessUnit", "CostCenter", "Department", "Division", "Owner", "Product", "Project"}

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

## Read parquets and create raw data table
df = spark.read \
    .format("parquet") \
    .option("recursiveFileLookup", "true") \
    .option("pathGlobFilter", "*.parquet") \
    .load(read_path_pattern)

# df = spark.sql("SELECT * FROM dbo.cost_raw")

# df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(export_table_raw)


StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 4, Finished, Available, Finished, False)

In [3]:
#helper functions:

def is_blank(col):
    return (col.isNull()) | (col == "") | (col == " ")

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 5, Finished, Available, Finished, False)

In [4]:
df = df.withColumn(
    "EffectiveCost",
    F.when(
        (F.col("ChargeCategory") == "Purchase") &
        (~is_blank(F.col("CommitmentDiscountId"))),
        F.lit(0.0)
    ).otherwise(F.col("EffectiveCost"))
)

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 6, Finished, Available, Finished, False)

In [5]:
df = df.select(
    *[
        F.col(c).cast("timestamp") if c in [
            "BillingPeriodEnd","BillingPeriodStart","ChargePeriodEnd","ChargePeriodStart",
            "x_BillingExchangeRateDate","x_ServicePeriodStart","x_ServicePeriodEnd"
        ]
        else F.col(c).cast("double") if c in [
            "BilledCost","ContractedCost","ContractedUnitPrice","EffectiveCost","ListCost","ListUnitPrice",
            "x_BilledCostInUsd","x_BilledUnitPrice","x_ContractedCostInUsd","x_EffectiveCostInUsd","x_EffectiveUnitPrice",
            "ConsumedQuantity","PricingQuantity","x_BillingExchangeRate","x_PartnerCreditRate","x_PricingBlockSize"
        ]
        else F.col(c)
        for c in df.columns
    ]
)

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 7, Finished, Available, Finished, False)

In [6]:
df = df.withColumn(
    "ContractedUnitPrice",
    F.when(F.col("ContractedUnitPrice") == 0,
        F.when(F.col("x_EffectiveUnitPrice") != 0, F.col("x_EffectiveUnitPrice"))
         .otherwise(F.col("ContractedUnitPrice"))
    ).otherwise(F.col("ContractedUnitPrice"))
)

df = df.withColumn(
    "ListUnitPrice",
    F.when(F.col("ListUnitPrice") == 0,
        F.when((F.col("ContractedUnitPrice") != 0) | (F.col("x_EffectiveUnitPrice") != 0),
               F.col("ContractedUnitPrice"))
         .otherwise(F.col("ListUnitPrice"))
    ).otherwise(F.col("ListUnitPrice"))
)

df = df.withColumn(
    "ContractedCost",
    F.when(F.col("ContractedCost") == 0,
        F.when(F.col("EffectiveCost") != 0, F.col("EffectiveCost"))
         .otherwise(F.col("ContractedCost"))
    ).otherwise(F.col("ContractedCost"))
)

df = df.withColumn(
    "ListCost",
    F.when(F.col("ListCost") == 0,
        F.when((F.col("ContractedCost") != 0) | (F.col("EffectiveCost") != 0),
               F.col("ContractedCost"))
         .otherwise(F.col("ListCost"))
    ).otherwise(F.col("ListCost"))
)

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 8, Finished, Available, Finished, False)

In [7]:
df = df.withColumn(
    "x_SkuDetailsDictionary",
    F.when(is_blank(F.col("x_SkuDetails")), F.map_from_arrays(F.array(), F.array()))
     .otherwise(F.from_json("x_SkuDetails", MapType(StringType(), StringType())))
)

df = df.withColumn(
    "x_TagsDictionary",
    F.when(is_blank(F.col("Tags")), F.map_from_arrays(F.array(), F.array()))
     .otherwise(F.from_json("Tags", MapType(StringType(), StringType())))
)

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 9, Finished, Available, Finished, False)

In [8]:
df = df \
.withColumn("x_CommitmentDiscountUtilizationPotential",
    F.when(F.col("CommitmentDiscountCategory")=="Usage", F.col("ConsumedQuantity"))
     .when(F.col("CommitmentDiscountCategory")=="Spend", F.col("EffectiveCost"))
     .otherwise(0)
) \
.withColumn("x_CommitmentDiscountUtilizationAmount",
    F.when(F.col("CommitmentDiscountStatus")=="Used",
           F.col("x_CommitmentDiscountUtilizationPotential")).otherwise(0)
) \
.withColumn("x_CommitmentDiscountSavings",
    F.when(is_blank(F.col("CommitmentDiscountCategory")), 0)
     .when(
        (F.col("ContractedCost")>0) &
        (F.col("EffectiveCost")>0) &
        (F.col("ContractedCost")>=F.col("EffectiveCost")),
        F.col("ContractedCost") - F.col("EffectiveCost")
     )
) \
.withColumn("x_NegotiatedDiscountSavings",
    F.when(
        (F.col("ListCost")>0) &
        (F.col("ContractedCost")>0) &
        (F.col("ListCost")>=F.col("ContractedCost")),
        F.col("ListCost") - F.col("ContractedCost")
    )
) \
.withColumn("x_AmortizationCategory",
    F.when(
        (F.col("ChargeCategory")=="Purchase") & (~is_blank(F.col("CommitmentDiscountId"))),
        "Principal"
    ).when(~is_blank(F.col("CommitmentDiscountId")), "Amortized Charge")
     .otherwise("")
) \
.withColumn("x_CommitmentDiscountPercent",
    F.when(F.col("ContractedUnitPrice")==0,0)
     .otherwise((F.col("ContractedUnitPrice")-F.col("x_EffectiveUnitPrice"))/F.col("ContractedUnitPrice"))
) \
.withColumn("x_NegotiatedDiscountPercent",
    F.when(F.col("ListUnitPrice")==0,0)
     .otherwise((F.col("ListUnitPrice")-F.col("ContractedUnitPrice"))/F.col("ListUnitPrice"))
) \
.withColumn("x_TotalDiscountPercent",
    F.when(F.col("ListUnitPrice")==0,0)
     .otherwise((F.col("ListUnitPrice")-F.col("x_EffectiveUnitPrice"))/F.col("ListUnitPrice"))
) \
.withColumn("x_TotalSavings",
    F.when(
        (F.col("ListCost")>0) & (F.col("EffectiveCost")>0) &
        (F.col("ListCost")>=F.col("EffectiveCost")),
        F.col("ListCost") - F.col("EffectiveCost")
    ).when(
        (F.col("ContractedCost")>0) & (F.col("EffectiveCost")>0) &
        (F.col("ContractedCost")>=F.col("EffectiveCost")),
        F.col("ContractedCost") - F.col("EffectiveCost")
    )
)

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 10, Finished, Available, Finished, False)

In [9]:
df = df \
.withColumn("x_ChargeMonth", F.date_trunc("month", F.col("ChargePeriodStart")).cast("date")) \
.withColumn("x_ReportingDate", F.to_date("ChargePeriodStart"))

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 11, Finished, Available, Finished, False)

In [10]:
for tag in PromotedTags:
    clean = "tag_" + tag.replace(":", "_").replace(" ", "")
    df = df.withColumn(clean, F.col("x_TagsDictionary").getItem(tag))

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 12, Finished, Available, Finished, False)

In [11]:
df = df \
.withColumn("tmp_VMName", F.col("x_SkuDetailsDictionary")["VMName"]) \
.withColumn("tmp_VMvCPUs", F.col("x_SkuDetailsDictionary")["VCPUs"].cast("long")) \
.withColumn("tmp_SQLvCores", F.col("x_SkuDetailsDictionary")["vCores"].cast("long")) \
.withColumn("tmp_SQLAHB", F.col("x_SkuDetailsDictionary")["AHB"]) \
.withColumn("x_SkuType", F.col("x_SkuDetailsDictionary")["ServiceType"])

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 13, Finished, Available, Finished, False)

In [12]:
df = df \
.withColumn(
    "x_ResourceMachineName",
    F.when(~is_blank(F.col("tmp_VMName")), F.col("tmp_VMName"))
) \
.withColumn(
    "x_SkuCoreCount",
    F.coalesce(F.col("tmp_VMvCPUs"), F.col("tmp_SQLvCores"))
) \
.withColumn(
    "x_ConsumedCoreHours",
    F.col("x_SkuCoreCount") * F.col("ConsumedQuantity")
) \
.withColumn(
    "x_SkuLicenseQuantity",
    F.when(F.col("x_SkuCoreCount").isNull(), 0)
     .when(F.col("x_SkuCoreCount") <= 8, 8)
     .otherwise(F.col("x_SkuCoreCount"))
)


StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 14, Finished, Available, Finished, False)

In [13]:
sku = F.col("x_SkuDetailsDictionary")

df = df.withColumn("x_SkuImageType", sku.getItem("ImageType")) \
       .withColumn("x_SkuUsageType", sku.getItem("UsageType")) \
       .withColumn("x_SkuType", sku.getItem("ServiceType")) \
       .withColumn("x_SkuMeterSubcategory", sku.getItem("MeterSubcategory"))

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 15, Finished, Available, Finished, False)

In [14]:
df = df.withColumn(
    "x_SkuLicenseStatus",
    F.when(
        (F.col("x_SkuMeterSubcategory").contains("Windows")) |
        (F.col("tmp_SQLAHB") == "False"),
        "Not enabled"
    ).when(
        (F.col("x_SkuImageType").contains("Windows Server BYOL")) |
        (F.col("tmp_SQLAHB") == "True") |
        (F.col("x_SkuMeterSubcategory").contains("Azure Hybrid Benefit")),
        "Enabled"
    ).otherwise("Not supported")
)

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 16, Finished, Available, Finished, False)

In [15]:
df = df.drop(
    "tmp_VMName","tmp_VMvCPUs","tmp_SQLvCores","tmp_SQLAHB","x_SkuDetailsDictionary","x_TagsDictionary"
)

df = df.select(sorted(df.columns))

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 17, Finished, Available, Finished, False)

In [16]:
df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(export_table_final)

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 18, Finished, Available, Finished, False)

In [17]:
from pyspark.sql import functions as F

# ----------------------------
# 1. min / max date
# ----------------------------
min_date = df.select(F.min("ChargePeriodStart")).first()[0]
max_date = df.select(F.max("ChargePeriodStart")).first()[0]

max_month_end = F.last_day(F.lit(max_date))

# ----------------------------
# 2. generate calendar
# ----------------------------
calendar_df = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{min_date}'),
            last_day(to_date('{max_date}')),
            interval 1 day
        )) AS Date
    """)
)

# ----------------------------
# 3. derived columns (ADDCOLUMNS equivalent)
# ----------------------------
calendar_df = calendar_df \
.withColumn("Year", F.year("Date")) \
.withColumn("Month No", F.month("Date")) \
.withColumn("Day", F.dayofmonth("Date")) \
.withColumn("Month", F.trunc("Date", "MM")) \
.withColumn("Is Weekend", F.when(F.dayofweek("Date").isin([1, 7]), 1).otherwise(0)) \
.withColumn("Is Last Date", F.when(F.col("Date") == F.lit(max_date), 1).otherwise(0)) \
.withColumn("Is Last Month",
    F.when(
        (F.month("Date") == F.month(F.lit(max_date))) &
        (F.year("Date") == F.year(F.lit(max_date))),
        1
    ).otherwise(0)
) \
.withColumn("Is Future Date",
    F.when(F.col("Date") > F.lit(max_date), 1).otherwise(0)
)


calendar_df.write.mode("overwrite").format("delta").option("delta.columnMapping.mode", "name").save(export_table_calendar)

StatementMeta(, 64c3815f-6d60-4a0d-a0cd-97078a901cd2, 19, Finished, Available, Finished, False)